In [1]:
from qiskit.circuit import Parameter, QuantumCircuit, QuantumRegister, ClassicalRegister

from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector, Operator
from scipy.optimize import minimize 
from qiskit.circuit.library import QFT
from qiskit import transpile
from qiskit.circuit.library import UnitaryGate

import random
import matplotlib.pyplot as plt
import scipy.linalg as scl
import numpy as np
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeKyiv
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
backend=AerSimulator()
# backend=FakeKyiv()
# sampler = Sampler(backend = backend)
pm = generate_preset_pass_manager(backend=backend,optimization_level=2)

In [3]:
I = np.array([[1,0],[0,1]])
X = np.array([[0,1],[1,0]])
Y = np.array([[0,-1j],[1j,0]])
Z = np.array([[1,0],[0,-1]])

m = np.array([[1, 0, 0, 0],
                 [0, 2, -1, 0],
                 [0, -1, 2, -1],
                 [0, 0, -1, 2]])
b = np.array([0,np.sqrt(2)/2,0.5,0.5])
nb_qubits = 2

def U_b(nb_qubits):
    circ = QuantumCircuit(nb_qubits)
    circ.prepare_state(b)
    return circ
U = U_b(nb_qubits)
b = np.array(Statevector(U_b(nb_qubits)))

def Hamiltonian(m):
    Ub = np.array(Operator(U_b(nb_qubits)))
    z = np.array([[1,0],
                 [0,-1]])
    I = np.array([[1,0],
                 [0,1]])
    def tensor(k,l):
        return np.kron(k,l)
    M1 = (np.dot(np.dot(Ub,tensor(z,I)),np.conj(Ub.T))
        + np.dot(np.dot(Ub,tensor(I,z)),np.conj(Ub.T)))
    M = 0.5*np.dot(np.dot(np.conj(m.T),(tensor(I,I) - M1/nb_qubits)),m)

    return M
A = Hamiltonian(m)

x_exact = np.linalg.solve(m,b)
x_exact = x_exact/np.linalg.norm(x_exact)
U1=scl.expm(2**0*2*np.pi*1j*A) 
U2=scl.expm(2**1*2*np.pi*1j*A) 
U3=scl.expm(2**2*2*np.pi*1j*A) 
U4=scl.expm(2**3*2*np.pi*1j*A) 

 
u1gate = UnitaryGate(U1)
u2gate = UnitaryGate(U2)
u3gate=UnitaryGate(U3)
u4gate=UnitaryGate(U4)



C_u1gate=u1gate.control()
C_u2gate=u2gate.control()
C_u3gate=u3gate.control()
C_u4gate=u4gate.control()
    

In [4]:

RMSE = []
nb_qubits = 2
depth = 2
qubits = list(range(nb_qubits))
N = len(qubits)
nb_params = int(9*N*depth)
RMSE = []
rep = 1000
for _ in range(rep):
    parameters = np.array([random.random() for _ in range(0, nb_params)])
    def ansatz(parameters):
        qc = QuantumCircuit(N)
        for d in range(depth):
            param1=parameters[d*9*N:(d+1)*(9*N)]
            for q in range(N):
                qc.ry(param1[q],qubits[q])
                qc.ry(param1[q+N],qubits[q])
                qc.ry(param1[q+2*N],qubits[q])
            qc.barrier()
            for q in range(N):
                qc.cx(qubits[q], qubits[(q+1)% N])
                qc.ry(param1[q+3*N],qubits[q])
                qc.ry(param1[q+4*N],qubits[(q+1)% N])
                qc.cx(qubits[(q+1)% N], qubits[q])
                qc.ry(param1[q+5*N],qubits[(q+1)% N])
                qc.cx(qubits[q], qubits[(q+1)% N])
            qc.barrier()
            if d==depth-1:
                for q in range(N):
                    qc.ry(param1[q+6*N],qubits[q])
                    qc.ry(param1[q+7*N],qubits[q])
                    qc.ry(param1[q+8*N],qubits[q])
            qc.barrier()
        
        return qc  
    
    def circ(parameters):
        x=QuantumRegister(6)
        c=ClassicalRegister(4)
        circuit = QuantumCircuit(x,c)
        phi=parameters
        circuit=circuit.compose(ansatz(parameters),x[4:6])
        circuit.h(x[0:4])

    #Apply the C-U operations where the controlled qubits are the qubits of the register x and the target from y    
        circuit.append(C_u1gate, [x[0],x[4],x[5]])    
        circuit.append(C_u2gate, [x[1],x[4],x[5]])    
        circuit.append(C_u3gate, [x[2],x[4],x[5]])    
        circuit.append(C_u4gate, [x[3],x[4],x[5]])     
    
    #Apply the inverse QFT on the x register   
        circuit &= QFT(num_qubits=4, approximation_degree=0, do_swaps=True, 
                       inverse=True, insert_barriers=False, name='qft')
        circuit.measure(x[0:4],c)       
        return circuit
    
    shots = 100000
    def cost(parameters):
        job = backend.run(pm.run(circ(parameters)),shots = shots).result()
        result = job.get_counts(0)
        res = 1 - result['0000']/shots
        return res
    # cost(parameters)
    
    def Optimizer(fun, x0, args=(), maxfev=None, reset_interval=None, eps=None, callback=None, **_):
        
        x0 = np.asarray(x0)
        recycle_z0 = None
        niter = 0
        funcalls = 0
    
        while True:

            idx = niter % x0.size
    
            if reset_interval > 0:
                if niter % reset_interval == 0:
                    recycle_z0 = None
    
            if recycle_z0 is None:
                z0 = fun(np.copy(x0), *args)
                funcalls += 1
            else:
                z0 = recycle_z0
    
            p = np.copy(x0)
            p[idx] = x0[idx] + np.pi / 2
            z1 = fun(p, *args)
            funcalls += 1
    
            p = np.copy(x0)
            p[idx] = x0[idx] - np.pi / 2
            z3 = fun(p, *args)
            funcalls += 1
    
            z2 = z1 + z3 - z0
            c = (z1 + z3) / 2
            a = np.sqrt((z0 - z2) ** 2 + (z1 - z3) ** 2) / 2
            b = np.arctan((z1 - z3) / ((z0 - z2) + 1e-32 * (z0 == z2))) + x0[idx]
            b += 0.5 * np.pi + 0.5 * np.pi * np.sign((z0 - z2) + eps * (z0 == z2))
            x0[idx] = b
            recycle_z0 = c - a
            if callback is not None:
                callback(np.copy(x0))
            if funcalls >= maxfev:
                break
            niter += 1
        # return OptimizeResult(fun=problabel0(np.copy(x0)), x=x0, nit=niter, 
        #                       nfev=funcalls, success=(niter > 1))
    
    def save(parameters):
        global Cost,Params
        Cost.append(cost(parameters))
        Params.append(parameters)
        # print(cost(parameters))
    Cost = []
    Params = []
    Optimizer(cost, parameters, args = (), maxfev = 2000, 
              reset_interval = 32, eps = 1e-32, callback=save)

    e = []
    F = []
    norm_e = []
    for k in range(len(Params)):
        state = np.array(Statevector(ansatz(Params[k])))
        norm = np.dot(state,x_exact)
        e.append(x_exact - state/norm)
        f = abs(np.dot(state,x_exact))**2
        F.append(f)

    for v in e:
        norm_e.append(float(np.linalg.norm(v)))
    Res = norm_e[np.argmax(F)]
    RMSE.append(Res)
print(RMSE)

[0.0001867860054656282, 0.00011043315594056171, 0.00012186048718894483, 0.00019405916442115495, 0.00014284501530893918, 0.0001318892508330484, 0.0001117703858759376, 0.00012654044473468554, 0.00010791608200096032, 0.00015259278196240973, 0.00014699305090181696, 0.0001674377802336898, 0.00015345134685210463, 0.00011593271943151165, 0.00012187043368576617, 0.00016780087860346257, 0.00010476516997130342, 0.0001411981854606532, 0.00012355291450601453, 0.000180463232512614, 0.00019500175827513359, 0.00018635351606431451, 0.0002001764106662433, 0.00010663805758936986, 0.00011361418119099125, 0.00010643027980442939, 0.00013099643041857405, 0.00011058883857441589, 0.000135704691345915, 0.0001255559336626865, 0.0001371332199452487, 0.0001325443689000364, 0.00016724520846282518, 0.0001298936312555492, 0.0001743022747603078, 0.0001615811856217329, 0.00013558603530889208, 0.00010970792412064807, 0.00017670227352962304, 0.00018455658280298892, 0.00011409016586937687, 0.00019294714118357973, 0.00014

In [5]:
float(np.mean(RMSE))

0.00015150997886230572